<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/xbrl-files/download-xsd-urls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to Find XSD URLs of Annual Reports on Form 10-K

This example demonstrates how to locate and download URLs for XSD files containing the XBRL taxonomy extension schema from 10-K filings submitted between 2020 and 2023. The located URLs are saved in a local file, `xsd_urls.csv`, and can be used to download the XSD files via the [Download API](https://sec-api.io/docs/sec-filings-render-api).

The link to the content of an XSD file can be found in the `documentUrl` field within the first item of the `dataFiles` array of a filing metadata object, as returned by the Query API.

The Python code example below shows how to use the Query API to search for 10-K filings within the specified date range and extract the URLs of the XSD files. Search queries are constructed for each month from 2020 to 2023, and all filings for that month are retrieved through pagination by incrementing the `from` parameter until all filings are collected. The XSD URLs are then extracted from the `dataFiles` array in the metadata and stored in a local CSV file.

The `QUICK_RUN` variable allows for a test run by skipping 90% of the URLs within the time range. Set it to `False` to retrieve all URLs.

In [ ]:
pip install sec-api

In [ ]:
from sec_api import QueryApi

queryApi = QueryApi(api_key="YOUR_API_KEY")

In [ ]:
# file to write XSD URLs to
XSD_FILE_NAME = "xsd_urls.csv"
# filing type to extract XSD URLs from
FILING_TYPE = "10-K"
# search range
YEAR_START = 2022
YEAR_END = 2023
# set True if you want to test the program and skip 90% of filings (quick run)
# set False if you want to download all URLs (takes up to 10 minutes)
QUICK_RUN = True

In [ ]:
search_params = {
  "query": "PLACEHOLDER", # this will be set during runtime
  "from": "0",
  "size": "50",
  "sort": [{ "filedAt": { "order": "desc" } }]
}

In [ ]:
# open the file we use to store the XSD URLs
log_file = open(XSD_FILE_NAME, "a")


# helper function to extract the XSD URL of the first item
# in the dataFiles list of a filing
def get_xsd_url(filing):
    if len(filing["dataFiles"]) > 0:
        return filing["dataFiles"][0]["documentUrl"]
    return None


for year in range(YEAR_START, YEAR_END + 1, 1):
    print(f"Starting {year}")

    # a single search universe is represented as a month of the given year
    for month in range(1, 13, 1):
        search_params["from"] = 0

        form_type_query = f'formType:"{FILING_TYPE}"'
        date_range_query = f"filedAt:[{year}-{month:02d}-01 TO {year}-{month:02d}-31]"
        # example query: "formType:\"10-K\" AND filedAt:[2021-01-01 TO 2021-01-31]"
        search_params["query"] = f"{form_type_query} AND {date_range_query}"

        print("Starting filing search for: ", search_params["query"])

        # paginate through results by increasing "from" parameter
        # until all filings are fetched
        end = 50 if QUICK_RUN else 10000
        for from_batch in range(0, end, 50):
            search_params["from"] = from_batch

            response = queryApi.get_filings(search_params)

            if len(response["filings"]) == 0:
                break

            # for each filing, get the URL of the XSD file
            urls_list = list(map(get_xsd_url, response["filings"]))

            # remove empty URLS
            urls_list = list(filter(None, urls_list))

            # transform list of URLs into one string by joining all list elements
            # and add a new-line character between each element.
            urls_string = "\n".join(urls_list) + "\n"

            log_file.write(urls_string)

            break


log_file.close()

print("All done!")

Starting 2022
Starting filing search for:  formType:"10-K" AND filedAt:[2022-01-01 TO 2022-01-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-02-01 TO 2022-02-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-03-01 TO 2022-03-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-04-01 TO 2022-04-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-05-01 TO 2022-05-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-06-01 TO 2022-06-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-07-01 TO 2022-07-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-08-01 TO 2022-08-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-09-01 TO 2022-09-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-10-01 TO 2022-10-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022-11-01 TO 2022-11-31]
Starting filing search for:  formType:"10-K" AND filedAt:[2022

In [ ]:
import pandas as pd

xsd_urls = pd.read_csv(XSD_FILE_NAME, header=None, names=["URL"])

print("Number of XSD File URLs:\n", xsd_urls.shape[0])
print("First 5 XSD File URLs:")
xsd_urls.head()

Number of XSD File URLs:
 120
First 5 XSD File URLs:


,URL
0,https://www.sec.gov/Archives/edgar/data/12239/...
1,https://www.sec.gov/Archives/edgar/data/831489...
2,https://www.sec.gov/Archives/edgar/data/106508...
3,https://www.sec.gov/Archives/edgar/data/109006...
4,https://www.sec.gov/Archives/edgar/data/12927/...
